# Introduction to Deep Learning, Assignment 2, Task 2


# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2025-12-17 12:10:53.447796: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 12:10:53.475717: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-17 12:11:00.533592: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


# Model with attention

### Pre training model

In [6]:
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [7]:
# Your code is: damn code

from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate,GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2, L1L2


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength, max_size=512):
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image


    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(X_in)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    #residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8), name = "residual_branch"))(B1_final)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=128, 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            #recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder

In [ ]:
def build_image2text_pretraining(dropout = 0.5, max_size=512, learning_rate = 2.5e-4,RLstrength=1.0e-4):

    vocab_size = 15

    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(6, vocab_size))

    encoder = build_image2text_encoder(dropout,RLstrength)
    _, hidden, cell = encoder(X_in)
    
    h_flattened = GlobalAveragePooling2D(name='h_flattened')(hidden)#Flatten()(hidden)
    h_initial = Dense(max_size, kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = GlobalAveragePooling2D(name='c_flattened')(cell)#Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]
    
    output, hidden, cell = LSTM(
        max_size, 
        return_sequences = True,
        return_state=True, 
        dropout = dropout, 
        #recurrent_dropout = dropout, 
        name='lstm_pre_training',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(Y_in, initial_state = ini_state)
        
    dense = TimeDistributed(Dense(vocab_size, activation='softmax', name = 'decoder_dense_pre_training'))
    y_out = dense(output)

    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'pretraining_model')
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    full.compile(
        loss=loss, optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
    )

    full.summary(expand_nested=True)
    return full



### Calculator model

In [9]:
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [10]:
# Here we build the full model
def build_text2text_calc(dropout = 0.5, max_size=512, learning_rate = 2.5e-4, max_answer_length_tf=4,RLstrength=1.0e-4):

    vocab_size = len(vocabulary_tf)

    # Define input layer of full model
    X_in = Input(shape = (6, vocab_size), name = 'expression_input')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)), name = "answer")


    # calculator encoder
    encoder_lstm = LSTM(max_size, return_state=True, return_sequences=True, name = 'calculator_encoder')
    key, hidden, cell = encoder_lstm(X_in)
    ini_state = [hidden, cell]    


    decoder_lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        ) #
    

    query, _, _ = decoder_lstm(Y_in, initial_state = ini_state)

    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = TimeDistributed(Activation('relu'), name = "decoder_activation")(final)
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    
    y_out = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'calculator')
    full.compile(
        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
    )

    full.summary(expand_nested=True)
    return full

### Concatenate models

In [11]:
from tensorflow.keras.layers import Lambda
import tensorflow.keras.backend as K

def build_image2text(pretraining_model, calculator_model):
    X_in = Input(shape=(5, 28, 28, 1), name='image_sequence')
    Y_in = Input(shape=(4, 15), name='answer_teacher_forcing')

    # This creates a (Batch, 6, 15) tensor of zeros 
    # compatible with the symbolic Keras graph.
    dummy_text = Lambda(lambda x: K.zeros_like(x[:, :1, :]))(Y_in) # Get (Batch, 1, 15)
    dummy_text = Lambda(lambda x: K.tile(x, [1, 6, 1]))(dummy_text) # Expand to (Batch, 6, 15)

    # Now this will work!
    expression_probs = pretraining_model([X_in, dummy_text])

    final_answer_probs = calculator_model([expression_probs, Y_in])

    full_pipeline = tf.keras.Model(inputs=[X_in, Y_in], outputs=final_answer_probs, name = "full_model")
    
    return full_pipeline

### Training models

In [ ]:
# Training the text interpreter. We train this before so that it's accuracy can be heightened

dropout=0.5
learning_rate=4.0e-4
RLstrength=1.0e-3
max_size=256
stopper_patience = 25
scheduler_patience = 5

image2text_pretraining = build_image2text_pretraining(dropout=dropout, learning_rate=learning_rate, RLstrength=RLstrength, max_size=max_size)

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 64,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper])


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_5  │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_26 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_27 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_28 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_29 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_30 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_32 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_33 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_34 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_35 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │      2,112 │ -                 │
│ time_distributed_31 │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/60


E0000 00:00:1765970233.782289 2123589 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid_2' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Si

250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - categorical_accuracy: 0.3852 - loss: 2.0324 - val_categorical_accuracy: 0.3528 - val_loss: 1.9061 - learning_rate: 4.0000e-04
Epoch 2/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step - categorical_accuracy: 0.4973 - loss: 1.5256 - val_categorical_accuracy: 0.4547 - val_loss: 1.6438 - learning_rate: 4.0000e-04
Epoch 3/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - categorical_accuracy: 0.5988 - loss: 1.2646 - val_categorical_accuracy: 0.4817 - val_loss: 1.5988 - learning_rate: 4.0000e-04
Epoch 4/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 13s 50ms/step - categorical_accuracy: 0.7086 - loss: 1.0070 - val_categorical_accuracy: 0.5255 - val_loss: 1.4504 - learning_rate: 4.0000e-04
Epoch 5/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.7929 - loss: 0.8114 - val_categorical_accuracy: 0.5873 - val_loss: 1.3926 - learning_rate: 4.0000e-04
Epoch 6/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.8479 - loss: 0.6713 -

In [15]:
history2 = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 64,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper])

Epoch 1/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.9902 - loss: 0.1311 - val_categorical_accuracy: 0.9685 - val_loss: 0.1982 - learning_rate: 1.0000e-04
Epoch 2/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.9904 - loss: 0.1270 - val_categorical_accuracy: 0.9633 - val_loss: 0.2107 - learning_rate: 1.0000e-04
Epoch 3/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - categorical_accuracy: 0.9905 - loss: 0.1236 - val_categorical_accuracy: 0.9436 - val_loss: 0.2624 - learning_rate: 1.0000e-04
Epoch 4/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.9899 - loss: 0.1231 - val_categorical_accuracy: 0.9534 - val_loss: 0.2379 - learning_rate: 1.0000e-04
Epoch 5/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - categorical_accuracy: 0.9903 - loss: 0.1212 - val_categorical_accuracy: 0.9609 - val_loss: 0.2090 - learning_rate: 1.0000e-04
Epoch 6/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - categorical_accuracy: 0.9906 - loss

In [16]:
# Training the calculator

dropout=0.5
learning_rate=5.0e-4
RLstrength=7.0e-4
max_size=256
stopper_patience = 25
scheduler_patience = 7

text2text_calculator = build_text2text_calc(dropout=dropout, learning_rate=learning_rate, RLstrength=RLstrength, max_size=max_size)

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 120,
               batch_size = 64,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper])


Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ answer[0][0],     │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - categorical_accuracy: 0.5247 - loss: 1.8601 - val_categorical_accuracy: 0.5598 - val_loss: 1.5752 - learning_rate: 5.0000e-04
Epoch 2/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.5670 - loss: 1.5039 - val_categorical_accuracy: 0.5786 - val_loss: 1.4218 - learning_rate: 5.0000e-04
Epoch 3/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.5954 - loss: 1.3361 - val_categorical_accuracy: 0.6274 - val_loss: 1.2288 - learning_rate: 5.0000e-04
Epoch 4/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.6277 - loss: 1.2017 - val_categorical_accuracy: 0.6408 - val_loss: 1.1381 - learning_rate: 5.0000e-04
Epoch 5/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.6518 - loss: 1.1074 - val_categorical_accuracy: 0.6614 - val_loss: 1.0614 - learning_rate: 5.0000e-04
Epoch 6/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.6690 - los

In [17]:
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

In [18]:
# Building full model
image2text =  build_image2text(image2text_pretraining, text2text_calculator)
image2text_pretraining.trainable = False

learning_rate_static = 5e-4

image2text.compile(optimizer=Adam(learning_rate=learning_rate_static), loss='categorical_crossentropy',metrics=['categorical_accuracy'])
image2text.summary(expand_nested=False)

# Train for few epochs with static visual encoder weights
history_full = image2text.fit(x=[X_train, y_train_in], y=y_train_target, 
               epochs = 7,
               batch_size = 32,
               validation_data = ([X_val, y_val_in], y_val_target))



# Further train with dynamic visual encoder weights
image2text_pretraining.trainable = True

stopper_patience = 25
scheduler_patience = 5
learning_rate_dynamic = 1.0e-5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

image2text.compile(optimizer=Adam(learning_rate=learning_rate_dynamic), loss='categorical_crossentropy',metrics=['categorical_accuracy'])
history_full = image2text.fit(x=[X_train, y_train_in], y=y_train_target, 
               epochs = 120,
               batch_size = 64,
               validation_data = ([X_val, y_val_in], y_val_target),
               callbacks=[lr_scheduler, early_stopper])

Model: "full_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer_teacher_for… │ (None, 4, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 15)     │          0 │ answer_teacher_f… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_sequence      │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 6, 15)     │          0 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretraining_model   │ (None, 6, 15)     │  1,254,991 │ image_sequence[0… │
│ (Functional)        │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator          │ (None, 4, 15)     │    696,591 │ pretraining_mode… │
│ (Functional)        │                   │            │ answer_teacher_f… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,951,582 (7.44 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 1,254,991 (4.79 MB)

Epoch 1/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - categorical_accuracy: 0.8793 - loss: 0.5767 - val_categorical_accuracy: 0.8451 - val_loss: 0.8777
Epoch 2/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.9351 - loss: 0.3303 - val_categorical_accuracy: 0.8558 - val_loss: 0.9124
Epoch 3/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.9394 - loss: 0.3254 - val_categorical_accuracy: 0.8616 - val_loss: 0.8789
Epoch 4/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.9422 - loss: 0.3109 - val_categorical_accuracy: 0.8669 - val_loss: 0.8479
Epoch 5/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.9455 - loss: 0.3002 - val_categorical_accuracy: 0.8642 - val_loss: 0.8371
Epoch 6/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.9479 - loss: 0.2892 - val_categorical_accuracy: 0.8643 - val_loss: 0.8290
Epoch 7/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - categorical_accuracy: 0.948

E0000 00:00:1765972654.596995 2123589 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_127/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_127/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_943/gradient_tape/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/full_model_1/pretr

250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 72ms/step - categorical_accuracy: 0.9676 - loss: 0.3207 - val_categorical_accuracy: 0.8796 - val_loss: 0.9062 - learning_rate: 1.0000e-05
Epoch 2/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9713 - loss: 0.3029 - val_categorical_accuracy: 0.8836 - val_loss: 0.8484 - learning_rate: 1.0000e-05
Epoch 3/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9722 - loss: 0.2993 - val_categorical_accuracy: 0.8867 - val_loss: 0.8235 - learning_rate: 1.0000e-05
Epoch 4/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9736 - loss: 0.2908 - val_categorical_accuracy: 0.8853 - val_loss: 0.8595 - learning_rate: 1.0000e-05
Epoch 5/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9733 - loss: 0.2919 - val_categorical_accuracy: 0.8839 - val_loss: 0.8552 - learning_rate: 1.0000e-05
Epoch 6/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9734 - loss: 0.2

#### Inference loop

In [ ]:
# Defining the inference procedure

def predict_math_expression(image_sequence, model, index_to_char, 
                            start_token='<start>', end_token='<end>', 
                            max_len=4, vocab_size=15):
    """
    image_sequence: Array of shape (5, 28, 28, 1)
    model: Your end-to-end image2text model
    index_to_char: Dictionary {int: char}
    """
    # 1. Prepare Inputs
    # Add batch dimension: (5, 28, 28, 1) -> (1, 5, 28, 28, 1)
    img_input = np.expand_dims(image_sequence, axis=0)
    
    # Initialize decoder input with zeros
    decoder_input = np.zeros((1, max_len, vocab_size))
    
    # Find the index for your start token
    # (Reverse the map if you only have char_to_index)
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    # Seed the first position with <start>
    decoder_input[0, 0, start_idx] = 1.0
    
    predicted_indices = [start_idx]
    
    # 2. Recursive Loop
    for i in range(max_len - 1):
        # Predict the next token
        preds = model.predict([img_input, decoder_input], verbose=0)
        
        # Get the character at the current step (i+1)
        next_idx = np.argmax(preds[0, i, :])
        predicted_indices.append(next_idx)
        
        # Stop if we hit the end token
        if next_idx == end_idx:
            break
            
        # Update the buffer for the next iteration
        if i + 1 < max_len:
            decoder_input[0, i + 1, next_idx] = 1.0

    # 3. Map to String
    # Join characters, skipping the special start/end tokens
    result_str = "".join([index_to_char[idx] for idx in predicted_indices 
                          if index_to_char[idx] not in [start_token, end_token]])
    
    return result_str

In [20]:
start = 3
stop = 10
sample_imgs = X_test[start: stop]
a = decode_labels_tf(y_test[start:stop])
expressions = []
for img in sample_imgs:
    expression = predict_math_expression(img, image2text, reverse_indices)
    expressions.append(expression)




In [21]:
print(a, expressions)

['62 ', '120', '-22', '91 ', '16 ', '13 ', '-17'] ['62 ', '119', '-22', '91 ', '16 ', '-7 ', '-17']


In [22]:
import numpy as np

def evaluate_calculator_performance(test_images, test_targets, model, index_to_char, 
                                   start_token='<start>', end_token='<end>', 
                                   max_len=4, vocab_size=15):
    """
    Scans a subset of the test set and calculates accuracy metrics.
    
    test_images: (N, 5, 28, 28, 1)
    test_targets: (N, 4, 15) one-hot encoded ground truth
    """
    total_samples = len(test_images)
    math_correct_count = 0
    total_tokens_correct = 0
    total_tokens_possible = 0
    
    # Pre-calculate reverse map for strings
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]

    print(f"Evaluating {total_samples} samples...")

    for i in range(total_samples):
        # 1. Generate Prediction using the Recursive Loop logic
        # (We use the logic from our previous 'predict_math_expression' function)
        img_in = np.expand_dims(test_images[i], axis=0)
        dec_in = np.zeros((1, max_len, vocab_size))
        dec_in[0, 0, start_idx] = 1.0
        
        pred_indices = [start_idx]
        for step in range(max_len - 1):
            p = model.predict([img_in, dec_in], verbose=0)
            next_idx = np.argmax(p[0, step, :])
            pred_indices.append(next_idx)
            if next_idx == end_idx: break
            if step + 1 < max_len:
                dec_in[0, step + 1, next_idx] = 1.0

        # 2. Extract Ground Truth indices
        true_indices = np.argmax(test_targets[i], axis=-1)

        # 3. Token Correctness (Character-by-character match)
        # We compare up to the length of the shorter sequence
        min_len = min(len(pred_indices), len(true_indices))
        for t in range(min_len):
            if pred_indices[t] == true_indices[t]:
                total_tokens_correct += 1
        total_tokens_possible += len(true_indices)

        # 4. Math Correctness (Full string match)
        # We clean the strings to compare actual values (e.g., "16" == "16")
        pred_str = "".join([index_to_char[idx] for idx in pred_indices 
                           if index_to_char[idx] not in [start_token, end_token, '<pad>']])
        true_str = "".join([index_to_char[idx] for idx in true_indices 
                           if index_to_char[idx] not in [start_token, end_token, '<pad>']])
        
        if pred_str == true_str:
            math_correct_count += 1

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{total_samples}...")

    # Final Metrics
    math_accuracy = math_correct_count / total_samples
    token_accuracy = total_tokens_correct / total_tokens_possible

    print("\n" + "="*30)
    print(f"RESULTS FOR {total_samples} SAMPLES")
    print(f"Token Accuracy: {token_accuracy:.2%}")
    print(f"Math Correctness: {math_accuracy:.2%}")
    print("="*30)

    return math_accuracy, token_accuracy

#### Inference

In [24]:
optimal_acc = 0.52 #choose your optimal thingetje

In [25]:
start = random.randint(0,1000)
N_samples = 100
X_sample_set = X_test[start: start+N_samples]
y_sample_set = y_test[start: start+N_samples]
math_accuracy, token_accuracy = evaluate_calculator_performance(X_sample_set, y_sample_set, model=image2text, index_to_char=reverse_indices)

loss, acc =image2text.evaluate(
    x=[X_train, y_train_in],
    y=y_train_target
)

val_loss, val_acc = image2text.evaluate(
    x=[X_val, y_val_in],
    y=y_val_target
)

print(f"dropout = {dropout}, RLstrength = {RLstrength} produces: math accuracy = {math_accuracy}, token accuracy = {token_accuracy}")
print(f"acc = {acc}, loss = {loss}")
print(f"val_acc = {val_acc}, val_loss = {val_loss}")

Evaluating 100 samples...
Processed 10/100...
Processed 20/100...
Processed 30/100...
Processed 40/100...
Processed 50/100...
Processed 60/100...
Processed 70/100...
Processed 80/100...
Processed 90/100...
Processed 100/100...

RESULTS FOR 100 SAMPLES
Token Accuracy: 71.20%
Math Correctness: 69.00%
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - categorical_accuracy: 0.9087 - loss: 0.6675
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - categorical_accuracy: 0.8947 - loss: 0.7538
dropout = 0.5, RLstrength = 0.0007 produces: math accuracy = 0.69, token accuracy = 0.712
acc = 0.9087286591529846, loss = 0.6675097346305847
val_acc = 0.8946776390075684, val_loss = 0.7537631392478943


#### Batch inference or something

In [27]:
import numpy as np

def evaluate_calculator_batch(test_images, test_targets, vision_model, calc_model, 
                              index_to_char, start_token='<start>', end_token='<end>', 
                              max_len=4, vocab_size=15):
    """
    vision_model: The part of the model that turns images into expression probs
    calc_model: The part that turns expression probs into answer tokens
    """
    num_samples = test_images.shape[0]
    
    # 1. Vision Phase: Process all images in one giant batch
    # We pass zeros for the teacher-forcing input of the vision part
    dummy_text = np.zeros((num_samples, 6, vocab_size))
    expression_probs = vision_model.predict([test_images, dummy_text], batch_size=32, verbose=1)
    
    # 2. Logic Phase: Recursive decoding in a batch loop
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    decoded_answer = np.zeros((num_samples, max_len, vocab_size))
    decoded_answer[:, 0, start_idx] = 1.0
    
    # Track which samples in the batch have hit the end token
    finished = np.zeros(num_samples, dtype=bool)
    final_indices = np.full((num_samples, max_len), end_idx)
    final_indices[:, 0] = start_idx

    for i in range(max_len - 1):
        # We only run the calculator model here (very fast)
        preds = calc_model.predict([expression_probs, decoded_answer], batch_size=num_samples, verbose=0)
        
        # Look at the prediction for the NEXT character
        next_indices = np.argmax(preds[:, i, :], axis=-1)
        
        for b_idx in range(num_samples):
            if not finished[b_idx]:
                token = next_indices[b_idx]
                final_indices[b_idx, i+1] = token
                decoded_answer[b_idx, i+1, token] = 1.0
                if token == end_idx:
                    finished[b_idx] = True
        
        if np.all(finished): break

    # 3. Calculation of Metrics
    true_indices = np.argmax(test_targets, axis=-1)
    
    token_correct = 0
    math_correct = 0
    
    for i in range(num_samples):
        p_str = "".join([index_to_char[idx] for idx in final_indices[i] if index_to_char[idx] not in [start_token, end_token, '<pad>']])
        t_str = "".join([index_to_char[idx] for idx in true_indices[i] if index_to_char[idx] not in [start_token, end_token, '<pad>']])
        
        # Math Accuracy
        if p_str == t_str:
            math_correct += 1
            
        # Token Accuracy
        for t_step in range(max_len):
            if final_indices[i, t_step] == true_indices[i, t_step]:
                token_correct += 1
                
    total_tokens = num_samples * max_len
    
    print(f"\nFinal Math Correctness: {math_correct/num_samples:.2%}")
    print(f"Final Token Accuracy: {token_correct/total_tokens:.2%}")
    
    return math_correct/num_samples, token_correct/total_tokens

In [ ]:
# Use the fast batch evaluator

X_sample_set = X_test
y_sample_set = y_test

math_acc, token_acc = evaluate_calculator_batch(
    X_sample_set, y_sample_set, 
    image2text_pretraining, text2text_calculator, 
    reverse_indices
)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step

Final Math Correctness: 71.30%
Final Token Accuracy: 89.68%


In [31]:
print(math_acc, token_acc)

0.713 0.89675


In [32]:
image2text.save(f'math_acc_{math_acc:3f}.keras')